# Breve introduzione a SQL e ai database relazionali

Queste note precedono la lezione su SQLite e spiegano *perché* esiste un DB relazionale, quali problemi risolve rispetto alle forme di serializzazione che già conosciamo (CSV, JSON), e quali astrazioni fondamentali introduce.

## Il problema con CSV (e JSON)

Un file CSV è una sequenza di righe omogenee. Va bene per una sola *entità* senza relazioni, ma fallisce appena ne aggiungiamo una seconda.

Supponiamo di avere paesi e città. Due opzioni:

**A) Un file unico (denormalizzato)**

<table>
<thead><tr><th>paese</th><th>continente</th><th>popolazione</th><th>città</th><th>abitanti</th></tr></thead>
<tbody>
<tr><td>Italia</td> <td>Europa</td><td>59 M</td>                                    <td>Roma</td>   <td>2.8 M</td></tr>
<tr><td>Italia</td> <td>Europa</td><td>59 M</td>                                    <td>Milano</td> <td>1.4 M</td></tr>
<tr><td>Italia</td> <td>Europa</td><td style="background:#fff176">62 M</td>         <td>Napoli</td> <td>1.0 M</td></tr>
<tr><td>Francia</td><td>Europa</td><td>68 M</td>                                    <td>Parigi</td> <td>2.1 M</td></tr>
</tbody>
</table>

La terza riga è stata aggiornata dimenticando di allineare le altre: il valore evidenziato è ormai inconsistente con le righe precedenti, e il formato non offre alcun meccanismo per rilevarlo.

**B) Due file separati**

<div style="display:flex; gap:3em">
<div>
<strong>paesi.csv</strong>
<table>
<thead><tr><th>codice</th><th>nome</th><th>continente</th><th>popolazione</th></tr></thead>
<tbody>
<tr><td>IT</td><td>Italia</td> <td>Europa</td><td>59 M</td></tr>
<tr><td>FR</td><td>Francia</td><td>Europa</td><td>68 M</td></tr>
</tbody>
</table>
</div>
<div>
<strong>citta.csv</strong>
<table>
<thead><tr><th>id</th><th>nome</th><th>paese_codice</th><th>abitanti</th></tr></thead>
<tbody>
<tr><td>1</td><td>Roma</td>  <td>IT</td><td>2800000</td></tr>
<tr><td>2</td><td>Milano</td><td>IT</td><td>1400000</td></tr>
</tbody>
</table>
</div>
</div>

Il "collegamento" tra i due file è solo una convenzione: nulla impedisce di inserire `paese_codice = 'XX'` inesistente. Il join tra i due file va scritto a mano in Python.

I database relazionali risolvono entrambi i problemi: formalizzano i collegamenti tramite **chiavi esterne** (foreign key) e delegano al motore la verifica dell'integrità e l'esecuzione dei join.

## Il modello relazionale in breve

Il modello relazionale nasce nel 1970 con un articolo di Edgar Codd (IBM). L'idea centrale è semplice:

- i dati sono organizzati in **tabelle** (relazioni): righe omogenee con colonne tipizzate
- ogni riga ha un **identificatore univoco** (chiave primaria)
- le tabelle si collegano tramite **chiavi esterne**: un valore in una tabella fa riferimento alla chiave primaria di un'altra
- un **linguaggio dichiarativo** (SQL) permette di interrogare e modificare i dati

```mermaid
erDiagram
    paesi {
        text    codice      PK
        text    nome
        text    continente
        integer popolazione
    }
    citta {
        integer id          PK
        text    nome
        text    paese_codice FK
        integer abitanti
    }
    paesi ||--o{ citta : "paese_codice"
```

La freccia simboleggia la *foreign key*: il motore rifiuta un inserimento che violerebbe il riferimento.

In [1]:
# Setup: creiamo il DB con cui lavorare negli esempi seguenti
import sqlite3

conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')

conn.executescript('''
CREATE TABLE paesi (
    codice      TEXT PRIMARY KEY,
    nome        TEXT NOT NULL,
    continente  TEXT,
    popolazione INTEGER
);

CREATE TABLE citta (
    id           INTEGER PRIMARY KEY,
    nome         TEXT NOT NULL,
    paese_codice TEXT NOT NULL REFERENCES paesi(codice),
    abitanti     INTEGER
);
''')

conn.executemany('INSERT INTO paesi VALUES (?,?,?,?)', [
    ('IT', 'Italia',   'Europa', 59_000_000),
    ('FR', 'Francia',  'Europa', 68_000_000),
    ('JP', 'Giappone', 'Asia',   125_000_000),
    ('BR', 'Brasile',  'America',215_000_000),
    ('NG', 'Nigeria',  'Africa', 220_000_000),
])
conn.executemany('INSERT INTO citta (nome, paese_codice, abitanti) VALUES (?,?,?)', [
    ('Roma',         'IT', 2_800_000),
    ('Milano',       'IT', 1_400_000),
    ('Napoli',       'IT', 1_000_000),
    ('Parigi',       'FR', 2_100_000),
    ('Marsiglia',    'FR',   870_000),
    ('Tokyo',        'JP',13_960_000),
    ('Osaka',        'JP', 2_700_000),
    ('San Paolo',    'BR',12_300_000),
    ('Rio de Janeiro','BR', 6_700_000),
    ('Lagos',        'NG',15_000_000),
    ('Kano',         'NG', 4_100_000),
])
conn.commit()
print('DB pronto.')

DB pronto.


### Il motore garantisce l'integrità

Con la foreign key attivata, un inserimento con un codice paese inesistente viene rifiutato:

In [2]:
try:
    conn.execute("INSERT INTO citta (nome, paese_codice, abitanti) VALUES ('Atlantide', 'XX', 0)")
except sqlite3.IntegrityError as e:
    print(f'Rifiutato: {e}')

Rifiutato: FOREIGN KEY constraint failed


### I tipi di dato: la garanzia mancante nei CSV

Un CSV non ha schema: ogni campo è testo. È l'applicazione a decidere come interpretarlo, e un'interpretazione sbagliata è invisibile al formato.

Un DB relazionale assegna un **tipo** a ogni colonna; il motore rifiuta i valori incompatibili e mantiene la garanzia ad ogni inserimento.

I tipi fondamentali di SQL:

| Tipo SQL | Significato |
|---|---|
| `INTEGER` | intero (fino a 64 bit) |
| `REAL` | virgola mobile IEEE 754 a 64 bit |
| `TEXT` | stringa Unicode |
| `BLOB` | dati binari arbitrari |
| `DECIMAL(p,s)` / `NUMERIC` | decimale **esatto** con *p* cifre totali e *s* decimali |

**Il caso dei dati finanziari**

`REAL` è un float IEEE 754. Come in Python, alcuni valori decimali non hanno rappresentazione esatta in binario — il classico esempio:

```python
0.1 + 0.2  # → 0.30000000000000004
```

Per importi, fatture e calcoli contabili questo è inaccettabile. Il tipo `DECIMAL`/`NUMERIC` dei database relazionali (PostgreSQL, MySQL, …) garantisce precisione arbitraria: il motore esegue l'aritmetica in base 10, non in virgola mobile.

> **Nota SQLite**: SQLite non implementa `DECIMAL` con aritmetica esatta — converte tutto a `REAL` o `INTEGER`. Per dati finanziari in SQLite la prassi è conservare gli importi come **interi in centesimi**, oppure delegare i calcoli a `decimal.Decimal` in Python.

In [ ]:
# Il problema: float IEEE 754
print(0.1 + 0.2)               # non è esattamente 0.3
print(0.1 + 0.2 == 0.3)        # False

# La soluzione in Python: decimal.Decimal
from decimal import Decimal
a, b = Decimal('0.1'), Decimal('0.2')
print(a + b)                    # 0.3
print(a + b == Decimal('0.3'))  # True

# SQLite: DECIMAL/NUMERIC è comunque mappato a REAL
demo = sqlite3.connect(':memory:')
demo.execute('CREATE TABLE t (v REAL, w NUMERIC)')
demo.execute('INSERT INTO t VALUES (0.1 + 0.2, 0.1 + 0.2)')
v, w, eq_v, eq_w = demo.execute('SELECT v, w, v = 0.3, w = 0.3 FROM t').fetchone()
print(f'REAL = {v}  ==0.3? {bool(eq_v)}')
print(f'NUMERIC = {w}  ==0.3? {bool(eq_w)}')
# → entrambi False: SQLite non ha aritmetica decimale esatta

# Prassi SQLite per valori monetari: centesimi come INTEGER
importo = Decimal('10.15')
centesimi = int(importo * 100)
print(f'{importo} € → {centesimi} centesimi (intero esatto)')
demo.close()

0.30000000000000004
False
0.3
True
REAL    = 0.30000000000000004  ==0.3? False
NUMERIC = 0.30000000000000004  ==0.3? False
10.15 € → 1015 centesimi (intero esatto)


## SQL è dichiarativo

La differenza fondamentale rispetto a un programma Python è che SQL descrive **cosa** si vuole ottenere, non **come** ottenerlo.

Esempio: "voglio nome della città, nome del paese e numero di abitanti, per le città con più di 2 milioni di abitanti, ordinate per popolazione decrescente."

```sql
SELECT c.nome AS città, p.nome AS paese, c.abitanti
FROM   citta c
JOIN   paesi p ON c.paese_codice = p.codice
WHERE  c.abitanti > 2000000
ORDER  BY c.abitanti DESC;
```

Non c'è alcuna istruzione su *come* realizzare il collegamento tra le due tabelle (nessun ciclo, nessun dizionario di supporto). È il **motore** a scegliere la strategia di esecuzione migliore — che può cambiare a seconda della dimensione dei dati, della presenza di indici, delle statistiche interne. Il codice SQL rimane invariato.

In Python il join manuale sarebbe:

```python
paesi_dict = {p['codice']: p['nome'] for p in paesi}
risultato = [
    (c['nome'], paesi_dict[c['paese_codice']], c['abitanti'])
    for c in citta
    if c['abitanti'] > 2_000_000
]
risultato.sort(key=lambda r: r[2], reverse=True)
```

Funziona, ma è codice imperativo che descrive *il come*: il programmatore sceglie la struttura dati (dizionario), la strategia (hash join), l'ordine delle operazioni.

In [4]:
# La stessa query in Python — il risultato è identico
for row in conn.execute('''
    SELECT c.nome AS città, p.nome AS paese, c.abitanti
    FROM   citta c
    JOIN   paesi p ON c.paese_codice = p.codice
    WHERE  c.abitanti > 2000000
    ORDER  BY c.abitanti DESC
'''):
    print(row)

('Lagos', 'Nigeria', 15000000)
('Tokyo', 'Giappone', 13960000)
('San Paolo', 'Brasile', 12300000)
('Rio de Janeiro', 'Brasile', 6700000)
('Kano', 'Nigeria', 4100000)
('Roma', 'Italia', 2800000)
('Osaka', 'Giappone', 2700000)
('Parigi', 'Francia', 2100000)


Un altro esempio di dichiaratività: le **aggregazioni**.

"Per ogni continente, numero di paesi e popolazione totale delle città censite:"

```sql
SELECT p.continente,
       COUNT(DISTINCT p.codice)  AS n_paesi,
       SUM(c.abitanti)           AS pop_citta
FROM   paesi p
JOIN   citta c ON c.paese_codice = p.codice
GROUP  BY p.continente
ORDER  BY pop_citta DESC;
```

Il motore decide se raggruppare *prima* o *dopo* il join, se usare tabelle hash o sort-based grouping. Noi dichiariamo solo il risultato atteso.

In [5]:
for row in conn.execute('''
    SELECT p.continente,
           COUNT(DISTINCT p.codice) AS n_paesi,
           SUM(c.abitanti)          AS pop_citta
    FROM   paesi p
    JOIN   citta c ON c.paese_codice = p.codice
    GROUP  BY p.continente
    ORDER  BY pop_citta DESC
'''):
    print(row)

('Africa', 1, 19100000)
('America', 1, 19000000)
('Asia', 1, 16660000)
('Europa', 2, 8170000)


## Gli indici e EXPLAIN QUERY PLAN

La dichiaratività funziona perché il motore dispone di strutture dati interne — gli **indici** — che rendono efficienti le operazioni di ricerca e join.

Un indice è, concettualmente, un **B-tree** costruito su una o più colonne: consente al motore di trovare le righe che soddisfano un predicato in tempo O(log n) invece di O(n).

L'istruzione `EXPLAIN QUERY PLAN` mostra quale strategia il motore ha scelto per eseguire una query. È uno strumento diagnostico, non esegue nulla.

In [6]:
# Senza indice: il motore deve scansionare tutta la tabella (SCAN)
print('--- senza indice ---')
for row in conn.execute('EXPLAIN QUERY PLAN SELECT * FROM citta WHERE abitanti > 2000000'):
    print(row)

--- senza indice ---
(2, 0, 0, 'SCAN citta')


In [7]:
# Con indice: il motore usa il B-tree (SEARCH / USING INDEX)
conn.execute('CREATE INDEX idx_abitanti ON citta(abitanti)')

print('--- con indice ---')
for row in conn.execute('EXPLAIN QUERY PLAN SELECT * FROM citta WHERE abitanti > 2000000'):
    print(row)

--- con indice ---
(3, 0, 0, 'SEARCH citta USING INDEX idx_abitanti (abitanti>?)')


Il punto chiave: la **query SQL non è cambiata**. È il motore che, rilevando la presenza dell'indice, ha scelto autonomamente una strategia diversa. Il codice applicativo è ignaro di questi dettagli.

Su tabelle di milioni di righe la differenza tra SCAN e SEARCH può essere di due o tre ordini di grandezza.

## Il motore: integrità, scalabilità, concorrenza

Un sistema di gestione di basi di dati (DBMS) non è solo un interprete SQL. Gestisce almeno tre problemi che altrimenti ricadrebbero sul codice applicativo:

### Integrità

Oltre alle foreign key già viste, il motore può imporre:
- **PRIMARY KEY**: unicità dell'identificatore
- **NOT NULL**: assenza di valori mancanti dove non ammessi
- **CHECK**: predicati arbitrari su una riga (`CHECK(voto BETWEEN 18 AND 30)`)
- **UNIQUE**: unicità di combinazioni di colonne
- **transazioni ACID**: un insieme di operazioni o viene eseguito *completamente* o viene annullato — non esistono stati intermedi visibili ad altri.

### Scalabilità

Un DBMS come PostgreSQL o MySQL gestisce tabelle con miliardi di righe distribuendo i dati su più file, usando buffer pool, statistiche adattive, e piani di esecuzione ottimizzati *a runtime*. Il codice applicativo rimane identico: si aggiunge un indice, si regola la configurazione del server, le query migliorano.

### Accessi concorrenti

Il motore gestisce l'isolamento tra transazioni concorrenti: due sessioni che leggono e scrivono contemporaneamente vedono stati coerenti, senza che il programmatore debba gestire lock espliciti nel codice applicativo.

---

### Dove si colloca SQLite?

SQLite implementa la grande maggioranza di questi meccanismi — integrità referenziale, transazioni ACID, indici B-tree, EXPLAIN — ma con un'architettura diversa dai DBMS client-server:

| Aspetto | PostgreSQL / MySQL | SQLite |
|---|---|---|
| Architettura | processo server separato | libreria nel processo dell'applicazione |
| Configurazione | richiesta (porta, utenti, …) | nessuna |
| Scalabilità scritture | alta (WAL, connessioni multiple) | limitata (un solo scrittore alla volta) |
| Concorrenza | gestita dal server | affidata al filesystem + WAL |
| Caso d'uso tipico | applicazioni multi-utente, web | dati locali, prototipazione, analisi |

SQLite è [usato ovunque](https://www.sqlite.org/mostdeployed.html): ogni smartphone ha centinaia di database SQLite (contatti, messaggi, preferenze delle app). È il formato di storage interno di Firefox, Chrome, e di molte applicazioni desktop. Il suo punto di forza non è la scalabilità ma la **semplicità di deployment** e la **robustezza** in scenari mono-utente o a bassa concorrenza.

## Riepilogo

| Concetto | In una riga |
|---|---|
| Tabella | sequenza di righe omogenee con colonne tipizzate e una chiave primaria |
| Foreign key | vincolo che formalizza il collegamento tra tabelle; il motore lo verifica |
| SQL dichiarativo | si descrive *cosa* si vuole, non *come* ottenerlo |
| Indice | struttura B-tree che rende O(log n) le ricerche; trasparente all'applicazione |
| EXPLAIN QUERY PLAN | mostra la strategia scelta dal motore (utile per ottimizzare) |
| Transazione ACID | gruppo di operazioni atomico, coerente, isolato, persistente |
| Tipo colonna | ogni colonna ha un tipo; il motore rifiuta valori incompatibili |
| `DECIMAL`/`NUMERIC` | aritmetica esatta per dati finanziari (in SQLite: usare interi in centesimi) |
| SQLite | DBMS embedded: tutto ciò che sopra, in un singolo file, senza server |